# Blazar Population ML — Project 1

**Goal:** Apply classical ML to the 4LAC-DR3 blazar catalog to recover constraints on jet baryon loading ($\mu$) from multi-wavelength observational features, and to characterize the blazar population via unsupervised clustering.

**Reference:** Rueda-Becerril, Harrison & Giannios (2021, MNRAS 501, 4092)

**Catalogs:**
- 4LAC-DR3: Ajello et al. (2022, arXiv:2209.12070), high-latitude, $|b| > 10^\circ$, `table-4LAC-DR3-h.fits`
- MOJAVE-XVII: apparent velocities from VLBI monitoring, `VizieR-MOJAVE-XVII.fit`

---

## Analysis Structure

- **Part 1 (full catalog, ~3,400 sources):** catalog-native features only, no kinematics
- **Part 2 (MOJAVE cross-matched, ~334 sources):** adds $\beta_{\text{app}}$ and $\Gamma_{\text{min}}$

**Notebook sections:**
1. Setup and imports
2. Load and inspect 4LAC-DR3
3. Load and inspect MOJAVE-XVII
4. Cross-match: 4LAC-DR3 × MOJAVE-XVII
5. Feature derivation and unified dataframes
6. Missingness audit
7. Exploratory data analysis (EDA)
8. Dimensionality reduction: PCA and UMAP
9. Clustering: GMM and HDBSCAN

## 1. Setup and Imports

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from astropy.io import fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

# Cosmology (consistent with 2021 paper)
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# Paths
DATA_DIR = Path('data/')
LAC_H  = DATA_DIR / 'table-4LAC-DR3-h.fits'
MOJAVE = DATA_DIR / 'VizieR-MOJAVE-XVII.fit'

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
CLASS_COLORS = {'bll': 'steelblue', 'fsrq': 'tomato', 'bcu': 'gray'}
SED_COLORS   = {'lsp': 'tomato', 'isp': 'goldenrod', 'hsp': 'steelblue'}

## 2. Load and Inspect 4LAC-DR3

Use `table-4LAC-DR3-h.fits` (high Galactic latitude, $|b| > 10^\circ$) as the primary catalog.
`table-4LAC-DR3-l.fits` covers low-latitude sources and is excluded from the main analysis.

**Tasks:**
- Print header data unit (HDU) structure and column list with units
- Check row count: expect 3,407 sources
- Print CLASS and SED_class distributions (note mixed-case entries: normalize to lowercase)
- Check for masked values vs. sentinel zeros, especially in `Redshift`, `nu_syn`, `nuFnu_syn`

### Inspecting `table-4LAC-DR3-h.fits`

In [ ]:
# Exploring header data unit (HDU)
with fits.open(LAC_H) as hdul:
    for i, hdu in enumerate(hdul):
        print(f"HDU {i}: {type(hdu).__name__}")
        if hasattr(hdu, 'columns') and hdu.columns:
            print(f"Rows: {len(hdu.data)}")
            print(f"Columns ({len(hdu.columns)}):")
            for col in hdu.columns:
                print(f"  {col.name:30s}  {col.format:6s}  {col.unit if col.unit else ''}")

HDU 0: PrimaryHDU
HDU 1: BinTableHDU
Rows: 3407
Columns (41):
  Source_Name                     18A     
  DataRelease                     I       
  RAJ2000                         E       deg
  DEJ2000                         E       deg
  GLON                            E       deg
  GLAT                            E       deg
  Signif_Avg                      E       
  Flux1000                        E       ph cm-2 s-1
  Unc_Flux1000                    E       ph cm-2 s-1
  Energy_Flux100                  E       erg cm-2 s-1
  Unc_Energy_Flux100              E       erg cm-2 s-1
  SpectrumType                    18A     
  PL_Index                        E       
  Unc_PL_Index                    E       
  Pivot_Energy                    E       MeV
  LP_Index                        E       
  Unc_LP_Index                    E       
  LP_beta                         E       
  Unc_LP_beta                     E       
  Flags                           I       
  CLASS          

### `CLASS` distribution

In [ ]:
t = Table.read(LAC_H)

# Finding unique values in CLASS column
classes, counts = np.unique([c.strip() for c in t['CLASS']], return_counts=True)

for cl, n in sorted(zip(classes, counts), key=lambda x: -x[1]):
    print(f"{cl:10s}  {n:5d}  ({100 * n / len(t):.1f}%)")

  bll          1357  (39.8%)
  bcu          1208  (35.5%)
  fsrq          715  (21.0%)
  FSRQ           40  (1.2%)
  rdg            36  (1.1%)
  BLL            22  (0.6%)
  RDG             6  (0.2%)
  agn             6  (0.2%)
  css             5  (0.1%)
  NLSY1           4  (0.1%)
  nlsy1           4  (0.1%)
  ssrq            2  (0.1%)
  AGN             1  (0.0%)
  sey             1  (0.0%)


### `SED_class` distribution

In [ ]:
def safe_strip(col):
    return [str(v).strip() if v is not np.ma.masked else '' for v in col]

sed_vals = safe_strip(t['SED_class'])
sed, counts = np.unique(sed_vals, return_counts=True)
for s, n in sorted(zip(sed, counts), key=lambda x: -x[1]):
    label = s if s else '(missing)'
    print(f"{label:10s}  {n:5d}  ({100 * n / len(t):.1f}%)")

   LSP          1564  (45.9%)
   (missing)     777  (22.8%)
   HSP           551  (16.2%)
   ISP           515  (15.1%)


### Missingness for key features

In [26]:
key_cols = [
    'PL_Index', 'LP_Index', 'Redshift', 'nu_syn', 'nuFnu_syn', 'HE_EPeak',
    'HE_nuFnuPeak', 'Variability_Index', 'Frac_Variability', 'Energy_Flux100'
]

for col in key_cols:
    arr = np.ma.filled(t[col].data, np.nan).astype(float)
    n_nan = np.sum(np.isnan(arr))
    n_zero = np.sum(arr == 0)
    print(f"{col:25s}  missing: {n_nan:4d} ({100 * n_nan / len(t):5.1f}%)  zero: {n_zero:4d}")
    
print(f"\nN = {len(t)}")

PL_Index                   missing:    0 (  0.0%)  zero:    0
LP_Index                   missing:    0 (  0.0%)  zero:    0
Redshift                   missing:    0 (  0.0%)  zero:    0
nu_syn                     missing:    0 (  0.0%)  zero:  777
nuFnu_syn                  missing:    0 (  0.0%)  zero:  777
HE_EPeak                   missing:  518 ( 15.2%)  zero:    0
HE_nuFnuPeak               missing:    0 (  0.0%)  zero:    0
Variability_Index          missing:    0 (  0.0%)  zero:    0
Frac_Variability           missing:    0 (  0.0%)  zero:  874
Energy_Flux100             missing:    0 (  0.0%)  zero:    0

N = 3407


### Derived features feasibility

In [ ]:
he = np.ma.filled(t['HE_nuFnuPeak'].data, np.nan).astype(float)
syn = np.ma.filled(t['nuFnu_syn'].data, np.nan).astype(float)
cd_valid = np.sum((~np.isnan(he)) & (~np.isnan(syn)) & (syn > 0) & (he > 0))
print(f"Compton dominance computable:    {cd_valid:4d}/{len(t)}  ({100 * cd_valid / len(t):.1f}%)")

z = np.ma.filled(t['Redshift'].data, np.nan).astype(float)
flux = np.ma.filled(t['Energy_Flux100'].data, np.nan).astype(float)
lum_valid = np.sum((~np.isnan(z)) & (z > 0) & (~np.isnan(flux)) & (flux > 0))
print(f"Gamma-ray luminosity computable: {lum_valid:4d}/{len(t)}  ({100 * lum_valid / len(t):.1f}%)")

Compton dominance computable:    2247/3407  (66.0%)
Gamma-ray luminosity computable: 1806/3407  (53.0%)


### `CLASS` case normalization issue

Mixed case entries detected: bll vs BLL, fsrq vs FSRQ; need normalization

In [ ]:
classes, counts = np.unique([c.strip().lower() for c in t['CLASS']], return_counts=True)

for cl, n in sorted(zip(classes, counts), key=lambda x: -x[1]):
    print(f"{cl:5s} + {cl.upper():5s} = {n:5d} ({100 * n / counts.sum():.1f}%)")

bll   + BLL   =  1379 (40.5%)
bcu   + BCU   =  1208 (35.5%)
fsrq  + FSRQ  =   755 (22.2%)
rdg   + RDG   =    42 (1.2%)
nlsy1 + NLSY1 =     8 (0.2%)
agn   + AGN   =     7 (0.2%)
css   + CSS   =     5 (0.1%)
ssrq  + SSRQ  =     2 (0.1%)
sey   + SEY   =     1 (0.0%)


## 3. Load and Inspect MOJAVE-XVII

The file has 6 HDUs. The relevant ones are:
- **HDU 1 (204 sources):** BL Lac-dominated. Columns include `ID`, `_RA`, `_DE`, `betaMax`, `e_betaMax`, `Cl`
- **HDU 2 (230 sources):** FSRQ/Quasar-dominated. Same key columns, also has `Smax` (peak flux)
- **HDU 4 (1,743 rows):** per-component kinematics — one row per tracked jet component, not used here

HDU 1 and HDU 2 have **zero overlap** in source IDs. Combine and deduplicate to get 434 unique sources.

**Warning:** The `SimbadName` column in HDUs 1 and 2 has a malformed FITS format that will crash `Table(hdul[i].data)`. Extract only the columns you need by iterating over `hdul[i].data` rows directly.

**Tasks:**
- Load HDU 1 and HDU 2, extract: `ID`, `_RA`, `_DE`, `betaMax`, `e_betaMax`, `Cl`
- Confirm zero overlap between HDU 1 and HDU 2 source IDs
- Combine and deduplicate on `ID`
- Report `betaMax` distribution; how many sources have betaMax = 0?

In [9]:
# Load and inspect MOJAVE-XVII


## 4. Cross-Match: 4LAC-DR3 × MOJAVE-XVII

Use `SkyCoord.match_to_catalog_sky` for positional matching.

- **4LAC-DR3 coords:** `RA_Counterpart` / `DEC_Counterpart` (counterpart position, more precise than LAT position). Filter out sources with masked counterpart coords before building the SkyCoord.
- **MOJAVE coords:** `_RA`, `_DE` (J2000, degrees)

Test tolerances 1", 2", 3", 5". If match count is stable across this range, 2" is safe — VLBI positions are precise enough that all real matches will be well under 1".

**Tasks:**
- Run cross-match, report match count and separation statistics
- Inspect CLASS and SED_class breakdown of matched sources
- Note the selection bias: MOJAVE selects radio-bright sources, mostly LSP/FSRQ

In [10]:
# Cross-match 4LAC-DR3 x MOJAVE-XVII


## 5. Feature Derivation and Unified Dataframes

Build two clean pandas DataFrames.

**Part 1 — all 3,407 sources, catalog-native + derived features:**

| Feature | Source column | Notes |
|---------|--------------|-------|
| `class` | `CLASS` | Lowercase, normalize |
| `sed_class` | `SED_class` | Lowercase |
| `redshift` | `Redshift` | Treat 0 as NaN |
| `pl_index` | `PL_Index` | Photon spectral index |
| `lp_alpha` | `LP_Index` | Log-parabola α |
| `lp_beta` | `LP_beta` | Log-parabola β (curvature) |
| `log_nu_syn` | log10(`nu_syn`) | Treat 0 as NaN |
| `log_nuFnu_syn` | log10(`nuFnu_syn`) | Treat 0 as NaN |
| `log_he_epeak` | log10(`HE_EPeak`) | MeV, treat 0 as NaN |
| `log_compton_dom` | log10(`HE_nuFnuPeak` × 1.602e-6 / `nuFnu_syn`) | Convert MeV/cm²/s → erg/cm²/s |
| `log_gamma_lum` | log10(4π d_L² × `Energy_Flux100`) | d_L from cosmo.luminosity_distance(z) |
| `var_index` | `Variability_Index` | |
| `frac_var` | `Frac_Variability` | Treat ≤ 0 as NaN |

**Part 2 — 334 MOJAVE-matched sources, adds:**

| Feature | Notes |
|---------|-------|
| `beta_app` | `betaMax` from MOJAVE, in units of c |
| `e_beta_app` | `e_betaMax` |
| `gamma_min` | sqrt(1 + β_app²) — lower bound on Lorentz factor; flag betaMax=0 sources |

Save both as parquet.

In [11]:
# Build Part 1 dataframe (all 4LAC-DR3-h sources)


In [12]:
# Build Part 2 dataframe (MOJAVE cross-matched sources)


In [13]:
# Save both dataframes


## 6. Missingness Audit

Document missingness for every feature. This drives feature set decisions in all downstream steps.

**Tasks:**
- Print % missing per feature for Part 1 and Part 2
- Plot a missingness heatmap (a sample of 500 rows is sufficient)
- Identify co-missing features: `nu_syn`=0 and `SED_class`=missing are the same 777 sources
- **Decision:** for Part 1 clustering, run with two feature sets: (a) SED features only (no luminosity), (b) SED + luminosity (redshift-complete subsample only). Compare results.

In [14]:
# Missingness audit — Part 1


In [15]:
# Missingness audit — Part 2


## 7. Exploratory Data Analysis

Understand the structure of the data before applying any ML. Any surprise here should change the modeling approach.

### 7.1 Univariate distributions
Histograms / KDE per feature, colored by CLASS (BLL=blue, FSRQ=red, BCU=gray). Look for bimodality, skewness, outliers.

### 7.2 BLL vs. FSRQ separation
Pairplot: `log_nu_syn`, `log_compton_dom`, `redshift`. The blazar sequence predicts BLLs at higher synchrotron peak, lower Compton dominance, and generally lower redshift than FSRQs.

### 7.3 Blazar sequence
Scatter: `log_nu_syn` vs. `log_compton_dom`, colored by SED_class (LSP/ISP/HSP). This is the canonical blazar sequence diagram — does the data reproduce it?

### 7.4 Correlation matrix
Pearson correlation heatmap on the complete-case subset. Flag strongly correlated pairs for the modeling phase.

In [16]:
# 7.1 Univariate distributions


In [17]:
# 7.2 BLL vs FSRQ pairplot


In [18]:
# 7.3 Blazar sequence: log_nu_syn vs log_compton_dom


In [19]:
# 7.4 Correlation matrix


## 8. Dimensionality Reduction: PCA and UMAP

**Feature set:** complete-case sources on core SED features: `pl_index`, `log_nu_syn`, `log_nuFnu_syn`, `log_compton_dom`, `var_index`. StandardScaler before both methods.

### 8.1 PCA
- Scree plot (explained variance ratio)
- PC1 vs PC2 colored by: CLASS, SED_class, log_nu_syn
- Loadings: does PC1 correspond to the blazar sequence?

### 8.2 UMAP
- `n_neighbors=15`, `min_dist=0.1`, `random_state=42`
- Plot colored by CLASS and SED_class
- Key question: where do BCU sources land relative to confirmed BLL and FSRQ?
- Install if needed: `pip install umap-learn`

In [20]:
# 8.1 PCA


In [21]:
# 8.2 UMAP


## 9. Clustering: GMM and HDBSCAN

**Goal:** let the data reveal its own groupings without imposing BLL/FSRQ labels. Then compare to known classifications.

### 9.1 Gaussian Mixture Model
- Fit for k = 2, 3, 4, 5 on the same feature set as Section 8
- Select k using BIC and AIC
- Plot cluster assignments on the UMAP embedding
- Confusion matrix: GMM clusters vs. BLL/FSRQ/BCU labels

### 9.2 HDBSCAN
- Fit on UMAP embedding (2D), not raw features
- Start with `min_cluster_size=30`
- Inspect noise points (label = -1): are they predominantly BCU?
- Install if needed: `pip install hdbscan`

### 9.3 BCU probabilistic classification
- Using the GMM posterior probabilities, assign soft class labels to BCU sources
- Report: how many BCUs fall clearly in one class? How many are genuinely intermediate?
- This answers the secondary scientific question

In [22]:
# 9.1 GMM


In [23]:
# 9.2 HDBSCAN


In [24]:
# 9.3 BCU probabilistic classification
